In [1]:
import numpy as np
from scipy.io import loadmat

from lib.read_data import u_ref, y_ref

MPCs = ["incremental", "IHMPC"]

mat = [loadmat(f"./MATLAB/{m}/output.mat") for m in MPCs]

u = np.array([mat[i]["uk"].T + u_ref for i in range(len(MPCs))])
y = np.array([mat[i]["yk"].T + y_ref for i in range(len(MPCs))])
y_sp = np.array([mat[i]["ysp"].T + y_ref for i in range(len(MPCs))])
Jk = np.array([mat[i]["Jk"].T for i in range(len(MPCs))])

cycles = np.linspace(0, u[0].shape[0] - 1, u[0].shape[0])

print(u.shape)


(2, 200, 4)


In [2]:
from lib.plot import colors, plt
from lib.read_data import u_labels, y_labels


def plot_control(cycles, u, y, sp, model_name: str):
    fig, axes = plt.subplots(3, 2, figsize=(16, 12), dpi=300, sharex=True)

    # t_feed
    axes[0, 0].plot(cycles, u[:, 0], label=u_labels[0])
    axes[0, 0].set_ylabel("Duração / s")

    # t_rinse
    axes[0, 1].plot(cycles, u[:, 1], label=u_labels[1])
    axes[0, 1].set_ylabel("Duração / s")

    # t_blow + t_purge
    axes[1, 0].plot(cycles, u[:, 2], label=u_labels[2])
    axes[1, 0].plot(cycles, u[:, 3], label=u_labels[3])
    axes[1, 0].set_ylabel("Duração / s")

    # Pureza H2
    axes[1, 1].plot(cycles, y[:, 0], label=y_labels[0], color=colors[0])
    axes[1, 1].plot(
        cycles, sp[:, 0], linestyle="--", color=colors[0], label=f"{y_labels[0]} (SP)"
    )
    axes[1, 1].set_ylabel("KPI / %")

    # Pureza CO2
    axes[2, 0].plot(cycles, y[:, 1], label=y_labels[1], color=colors[1])
    axes[2, 0].plot(
        cycles, sp[:, 1], linestyle="--", color=colors[1], label=f"{y_labels[1]} (SP)"
    )
    axes[2, 0].set_ylabel("KPI / %")

    # Recuperação CO2
    axes[2, 1].plot(cycles, y[:, 2], label=y_labels[2], color=colors[2])
    axes[2, 1].plot(
        cycles, sp[:, 2], linestyle="--", color=colors[2], label=f"{y_labels[2]} (SP)"
    )
    axes[2, 1].set_ylabel("KPI / %")

    for ax in axes.flat:
        ax.grid(True)
        ax.legend()
        ax.set_xlabel("Ciclo")

    plt.tight_layout()
    plt.savefig(f"../figures/{model_name}.png")
    plt.close()


for i in range(len(MPCs)):
    plot_control(cycles, u[i], y[i], y_sp[i], MPCs[i])


In [3]:
from lib.plot import legend_outside

_, axes = plt.subplots(3, 1, sharex=True, figsize=(16, 7))

for i in range(len(y_labels)):
    for j in range(len(MPCs)):
        axes[i].plot(cycles, y[j, :, i], label=f"{y_labels[i]} ({MPCs[j]})")
    axes[i].plot(
        cycles,
        y_sp[0, :, i],
        linestyle="--",
        label=f"{y_labels[i]} (SP)",
        color="tab:red",
    )

axes[-1].set_xlabel("Ciclo")
for ax in axes:
    legend_outside(ax)
    ax.grid(True)
    ax.set_ylabel("KPIs / %")


plt.tight_layout()
plt.savefig("../figures/compare-MPCs.png")
plt.close()


In [4]:
_, axes = plt.subplots(2, 1, sharex=True, figsize=(12, 8))

for i in range(len(MPCs)):
    axes[i].plot(cycles, Jk[i], label=f"{MPCs[i]}")

axes[-1].set_xlabel("Ciclo")
for ax in axes:
    ax.legend()
    ax.grid(True)
    ax.set_ylabel("Função Custo")

plt.savefig("../figures/cost-functions.png")
plt.close()


# Métricas


In [5]:
e = y - y_sp

# Integral Absolute Error
IAE = np.sum(np.abs(e), axis=1)

# Integral Square Error
ISE = np.sum(e**2, axis=1)


def print_metric_table(metric, metric_name, controllers):
    header = ["Controlador", *y_labels, "Total"]

    n_cols = len(header)
    row_fmt = "{:<14}" + " {:>22}" * (n_cols - 2) + " {:>15}"

    print(f"\n{metric_name}")
    print(row_fmt.format(*header))
    print("-" * (14 + 22 * (n_cols - 2) + 15 + (n_cols - 1)))

    for name, values in zip(controllers, metric):
        total = sum(values)
        print(
            row_fmt.format(
                name,
                *[f"{v:.4f}" for v in values],
                f"{total:.4f}",
            )
        )


print_metric_table(IAE, "IAE", MPCs)
print_metric_table(ISE, "ISE", MPCs)



IAE
Controlador           Pureza do H$_2$       Pureza do CO$_2$  Recuperação de CO$_2$           Total
---------------------------------------------------------------------------------------------------
incremental                   26.3969                45.2334                22.6979         94.3282
IHMPC                         12.0353                32.1269                28.6527         72.8149

ISE
Controlador           Pureza do H$_2$       Pureza do CO$_2$  Recuperação de CO$_2$           Total
---------------------------------------------------------------------------------------------------
incremental                   10.3547               106.6367                36.6850        153.6764
IHMPC                          6.2921                77.6338                56.2245        140.1505
